In [1]:
%pip install -U pandas numpy pymatgen tqdm

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import ast
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

from pymatgen.core import Composition


BASE_DIR = Path("sodium_cathode_cde_matching")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "outputs"
AUDIT_DIR = BASE_DIR / "audit"

for folder in [BASE_DIR, INPUT_DIR, OUTPUT_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE_DIR.resolve())

Project folder: C:\Users\IICT-1\next sustainability\code 2\sodium_cathode_cde_matching


In [3]:
def find_first_existing(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError(
        "None of these files were found:\n" + "\n".join(str(x) for x in candidates)
    )


MP_FINAL_PATH = find_first_existing([
    Path("mp_na_electrodes_conservative_candidates_v1.csv"),
    Path("mp_sodium_battery_dataset/final/mp_na_electrodes_conservative_candidates_v1.csv"),
    Path("/mnt/data/mp_na_electrodes_conservative_candidates_v1.csv"),
])

MP_RANKED_PATH = find_first_existing([
    Path("mp_na_electrodes_ranked_preliminary.csv"),
    Path("mp_sodium_battery_dataset/corrected/mp_na_electrodes_ranked_preliminary.csv"),
    Path("/mnt/data/mp_na_electrodes_ranked_preliminary.csv"),
])

BATTERY_ZIP_PATH = find_first_existing([
    Path("Battery Data.zip"),
    Path("battery_data.zip"),
    Path("data/raw/Battery Data.zip"),
    Path("/mnt/data/Battery Data.zip"),
])

MP_METADATA_PATH = find_first_existing([
    Path("materials_project_download_metadata.json"),
    Path("mp_sodium_battery_dataset/metadata/materials_project_download_metadata.json"),
    Path("/mnt/data/materials_project_download_metadata.json"),
])

print("MP final candidate file:", MP_FINAL_PATH)
print("MP ranked file:", MP_RANKED_PATH)
print("CDE battery zip:", BATTERY_ZIP_PATH)
print("MP metadata:", MP_METADATA_PATH)

MP final candidate file: mp_na_electrodes_conservative_candidates_v1.csv
MP ranked file: mp_na_electrodes_ranked_preliminary.csv
CDE battery zip: Battery Data.zip
MP metadata: materials_project_download_metadata.json


In [4]:
df_mp = pd.read_csv(MP_FINAL_PATH)
df_mp_ranked = pd.read_csv(MP_RANKED_PATH)

with open(MP_METADATA_PATH, "r", encoding="utf-8") as f:
    mp_metadata = json.load(f)

print("MP final shape:", df_mp.shape)
print("MP ranked shape:", df_mp_ranked.shape)
print("MP database version:", mp_metadata.get("mp_database_version"))

for col in [
    "basic_screening_pass",
    "hard_exclusion_free_candidate",
    "conservative_earth_abundant_candidate",
    "preliminary_family",
]:
    if col in df_mp.columns:
        print("\n", col)
        display(df_mp[col].value_counts(dropna=False).to_frame("count"))

display(df_mp.head())

MP final shape: (416, 230)
MP ranked shape: (416, 224)
MP database version: 2026.04.13

 basic_screening_pass


,count
basic_screening_pass,
False,252
True,164



 hard_exclusion_free_candidate


,count
hard_exclusion_free_candidate,
False,304
True,112



 conservative_earth_abundant_candidate


,count
conservative_earth_abundant_candidate,
False,360
True,56



 preliminary_family


,count
preliminary_family,
phosphate_or_NASICON_like,141
transition_metal_oxide_like,122
other_or_unclear,107
sulfate_like,23
silicate_like,18
carbon_or_organic_like,5


,formula_discharge,working_ion,material_ids,formula_charge,fracA_charge,nelements,max_delta_volume,framework_formula,energy_grav,num_steps,...,volume_score,stability_score,criticality_score,mp_preliminary_score,element_set_v2,hard_exclusion_v2_elements,soft_penalty_v2_elements,hard_exclusion_v2_count,soft_penalty_v2_count,conservative_earth_abundant_candidate
0,Na2MnPO4F,Na,"[""mp-aaabxacc"", ""mp-aaabfrce""]",MnPO4F,0.000000,4,0.286562,MnPO4F,964.105080,1,...,0.992836,0.823462,1.0,0.909658,"""{'O', 'F', 'P', 'Na', 'Mn'}""",NaN,NaN,0,0,True
1,Na2MnP2O7,Na,"[""mp-aaabhhug"", ""mp-aaacbgbk"", ""mp-aaacqeli""]",MnP2O7,0.000000,3,0.123101,MnP2O7,791.536945,1,...,0.996922,0.636887,1.0,0.895030,"""{'O', 'P', 'Mn', 'Na'}""",NaN,NaN,0,0,True
2,Na2MnCSO7,Na,"[""mp-aaabqyiz"", ""mp-aaabrxto""]",MnCSO7,0.000000,4,0.212925,MnCSO7,855.363301,1,...,0.994677,0.563417,1.0,0.894334,"""{'O', 'S', 'Na', 'Mn', 'C'}""",NaN,NaN,0,0,True
3,Na3MnPCO7,Na,"[""mp-aaabrhtw"", ""mp-aaabrsfg"", ""mp-aaabsfnd"", ...",NaMnPCO7,0.090909,4,0.073266,MnPCO7,647.638332,2,...,0.998168,1.000000,1.0,0.877118,"""{'O', 'P', 'Na', 'Mn', 'C'}""",NaN,NaN,0,0,True
4,Na4Fe2(PO4)3,Na,"[""mp-aaaabnik"", ""mp-aaaabnnc"", ""mp-aaaabllc"", ...",Fe2(PO4)3,0.000000,3,0.036937,Fe2(PO4)3,868.190282,2,...,0.999077,0.519503,1.0,0.873535,"""{'O', 'P', 'Na', 'Fe'}""",NaN,NaN,0,0,True


In [5]:
with zipfile.ZipFile(BATTERY_ZIP_PATH) as zf:
    print("Files inside zip:")
    for name in zf.namelist():
        print(" -", name)

    with zf.open("battery_merged.csv") as f:
        df_cde = pd.read_csv(f)

print("CDE merged database shape:", df_cde.shape)
print("Columns:")
print(df_cde.columns.tolist())

display(df_cde.head())

Files inside zip:
 - battery.csv
 - battery.db
 - battery.json
 - battery_merged.csv
 - battery_merged.db
 - battery_merged.json
CDE merged database shape: (214617, 18)
Columns:
['Property', 'Name', 'Value', 'Raw_unit', 'Raw_value', 'Unit', 'Num_records', 'Extracted_name', 'DOI', 'Specifier', 'Tag', 'Warning', 'Type', 'Info', 'Title', 'Journal', 'Date', 'Correctness']


,Property,Name,Value,Raw_unit,Raw_value,Unit,Num_records,Extracted_name,DOI,Specifier,Tag,Warning,Type,Info,Title,Journal,Date,Correctness
0,Capacity,Fe2O3 / C,659.0,mAhg−1,and 659,Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),1,"[{'Fe': '2.0', 'O': '3.0'}, {'C': '1.0'}]",10.1016/j.jallcom.2011.10.022,"discharge and charge capacity,",CDE,"S,",NaN,NaN,SOLVOTHERMALPREPARATIONLITHIUMSTORAGEPROPERTIE...,Journal of Alloys and Compounds,2011-10-17,NaN
1,Coulombic Efficiency,Ni @ C,100.0,%,100,Percent^(1.0),1,"[{'Ni': '1.0'}, {'C': '1.0'}]",10.1039/C6TA02339H,"coulombic,",CDE,NaN,NaN,NaN,Mesoporous Ni@C hybrids for a high energy aque...,Journal of Materials Chemistry A,2016/06/14,NaN
2,Voltage,Li2O2,2.3,V,2.3,Volt^(1.0),4,"[{'Li': '2.0', 'O': '2.0'}]","10.1016/j.electacta.2015.06.071, 10.1039/C6TA0...","voltage, voltage,",CDE,"R,",NaN,NaN,"CAPACITYENHANCEMENTALITHIUMOXYGENFLOWBATTERY, ...","Electrochimica Acta, Journal of Materials Chem...","2015-06-24, 2016/06/07, 2015/07/01, 2019-09-18",NaN
3,Capacity,Li2C8H4O4,85.0,mAhg−1,85,Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),1,"[{'Li': '2.0', 'C': '8.0', 'H': '4.0', 'O': '4...",10.1016/j.pnsc.2016.06.004,"charge capacity,",CDE,"S,",NaN,"{'current_value': '0.2', 'current_units': 'C'},",ADESIGNEDCORESHELLSTRUCTURALCOMPOSITELITHIUMTE...,Progress in Natural Science: Materials Interna...,2016-08-10,NaN
4,Capacity,NVP / C,83.0,mAhg−1,"96.6 , 90.5 , 86.8 , 83.0 , 77.1 , 68.0 , and ...",Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),1,"[{'N': '1.0', 'V': '1.0', 'P': '1.0'}, {'C': '...",10.1016/j.ensm.2019.05.041,"discharge capacities,",CDE,"S,",NaN,"{'current_value': '1', 'current_units': 'C'},",UNDERSTANDINGSUPERIORSODIUMIONSTORAGEINANOVELN...,Energy Storage Materials,2019-06-01,NaN


In [6]:
def formula_to_reduced(formula):
    """
    Convert formula string to pymatgen reduced formula.
    Returns None if parsing fails.
    """
    if pd.isna(formula):
        return None

    s = str(formula).strip()

    if not s or s.lower() in {"nan", "none", "null"}:
        return None

    try:
        return Composition(s).reduced_formula
    except Exception:
        return None


def formula_elements(formula):
    """
    Return element symbols from formula.
    """
    if pd.isna(formula):
        return set()

    try:
        return set(el.symbol for el in Composition(str(formula)).elements)
    except Exception:
        return set()


def safe_literal_eval(x):
    """
    Safely parse CDE Extracted_name field, which is usually a stringified list of dictionaries.
    """
    if pd.isna(x):
        return None

    try:
        return ast.literal_eval(str(x))
    except Exception:
        return None


def compdict_to_reduced(d):
    """
    Convert CDE dictionary like {'Na': '3.0', 'V': '2.0', 'P': '3.0', 'O': '12.0'}
    into reduced formula.
    """
    try:
        comp = {}

        for k, v in d.items():
            if k is None:
                continue

            val = float(v)

            if val > 0:
                comp[str(k)] = val

        if not comp:
            return None

        return Composition(comp).reduced_formula

    except Exception:
        return None


FORMULA_TOKEN_RE = re.compile(
    r"([A-Z][a-z]?(?:\d*\.?\d*)?(?:[A-Z][a-z]?(?:\d*\.?\d*)?)+(?:\([A-Za-z0-9.]+\)\d*\.?\d*)*)"
)


def extract_formula_candidates_from_name(name):
    """
    Try to extract formula-like strings from the CDE Name field.
    This is supplementary because CDE Name can contain abbreviations such as NVP/C.
    """
    if pd.isna(name):
        return []

    text = str(name)
    parts = re.split(r"[/@·,;\s]+", text)

    candidates = []

    for part in parts + FORMULA_TOKEN_RE.findall(text):
        part = str(part).strip().strip("()[]{}")

        if not part:
            continue

        red = formula_to_reduced(part)

        if red is not None:
            candidates.append(red)

    # Preserve order and remove duplicates
    unique = []
    for f in candidates:
        if f not in unique:
            unique.append(f)

    return unique


def extract_cde_formulas(row):
    """
    Extract all formula candidates from CDE row using:
    1. Extracted_name
    2. Name field fallback
    """
    formulas = []

    parsed = safe_literal_eval(row.get("Extracted_name"))

    if isinstance(parsed, list):
        combined = {}

        for item in parsed:
            if isinstance(item, dict):
                red = compdict_to_reduced(item)

                if red:
                    formulas.append(red)

                # Also store combined composite formula as a weak fallback
                try:
                    for k, v in item.items():
                        combined[k] = combined.get(k, 0) + float(v)
                except Exception:
                    pass

        red_combined = compdict_to_reduced(combined) if combined else None

        if red_combined:
            formulas.append(red_combined)

    formulas.extend(extract_formula_candidates_from_name(row.get("Name")))

    unique = []
    for f in formulas:
        if f and f not in unique:
            unique.append(f)

    return unique


def split_doi_list(x):
    if pd.isna(x):
        return []

    return [
        d.strip()
        for d in str(x).split(",")
        if d.strip() and d.strip().lower() != "nan"
    ]


def has_warning_flag(x, flags):
    if pd.isna(x):
        return False

    text = str(x).upper()

    return any(flag.upper() in text for flag in flags)

In [7]:
required_cols = [
    "Property",
    "Name",
    "Value",
    "Unit",
    "Raw_value",
    "Raw_unit",
    "Warning",
    "Type",
    "DOI",
    "Title",
    "Journal",
    "Date",
    "Tag",
    "Info",
    "Specifier",
    "Num_records",
    "Extracted_name",
]

for col in required_cols:
    if col not in df_cde.columns:
        df_cde[col] = np.nan

df_cde["Value"] = pd.to_numeric(df_cde["Value"], errors="coerce")
df_cde["Num_records"] = pd.to_numeric(df_cde["Num_records"], errors="coerce").fillna(1)

df_cde["property_clean"] = df_cde["Property"].astype(str).str.strip()

df_cde["warning_has_L_or_R"] = df_cde["Warning"].map(
    lambda x: has_warning_flag(x, ["L", "R"])
)

df_cde["warning_has_S"] = df_cde["Warning"].map(
    lambda x: has_warning_flag(x, ["S"])
)

df_cde["warning_clean_no_LR"] = ~df_cde["warning_has_L_or_R"]

context_cols = ["Name", "Title", "Type", "Specifier"]

combined_context = (
    df_cde[context_cols]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.lower()
)

df_cde["sodium_text_context"] = combined_context.str.contains(
    r"\bna\b|sodium|sodium[- ]ion|na[- ]ion",
    regex=True,
)

df_cde["cathode_text_context"] = combined_context.str.contains(
    r"cathode|positive electrode",
    regex=True,
)

df_cde["anode_or_electrolyte_context"] = combined_context.str.contains(
    r"anode|negative electrode|electrolyte|separator",
    regex=True,
)

print("CDE rows:", len(df_cde))
print("Warning L/R rows:", int(df_cde["warning_has_L_or_R"].sum()))
print("Warning S rows:", int(df_cde["warning_has_S"].sum()))
print("Sodium text-context rows:", int(df_cde["sodium_text_context"].sum()))
print("Cathode text-context rows:", int(df_cde["cathode_text_context"].sum()))

display(df_cde["property_clean"].value_counts().to_frame("count"))

CDE rows: 214617
Warning L/R rows: 46746
Warning S rows: 99859
Sodium text-context rows: 25955
Cathode text-context rows: 37511


,count
property_clean,
Capacity,113171
Voltage,71461
Energy,14272
Coulombic Efficiency,9874
Conductivity,5839


In [8]:
tqdm.pandas()

df_cde["cde_formula_candidates"] = df_cde.progress_apply(
    extract_cde_formulas,
    axis=1,
)

df_cde["formula_candidate_count"] = df_cde["cde_formula_candidates"].map(len)

print("Rows with at least one parsed formula:", int((df_cde["formula_candidate_count"] > 0).sum()))
print("Rows without parsed formula:", int((df_cde["formula_candidate_count"] == 0).sum()))

display(df_cde[["Name", "Extracted_name", "cde_formula_candidates"]].head(20))

  0%|          | 0/214617 [00:00<?, ?it/s]

C:\Users\IICT-1\miniconda3\envs\myenv\Lib\functools.py:1001: UserWarning: No Pauling electronegativity for Ts. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)
C:\Users\IICT-1\miniconda3\envs\myenv\Lib\functools.py:1001: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)
C:\Users\IICT-1\miniconda3\envs\myenv\Lib\functools.py:1001: UserWarning: No Pauling electronegativity for Ds. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)
C:\Users\IICT-1\miniconda3\envs\myenv\Lib\functools.py:1001: UserWarning: No Pauling electronegativity for Mt. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a flo

Rows with at least one parsed formula: 213260
Rows without parsed formula: 1357


,Name,Extracted_name,cde_formula_candidates
0,Fe2O3 / C,"[{'Fe': '2.0', 'O': '3.0'}, {'C': '1.0'}]","[Fe2O3, C, Fe2CO3]"
1,Ni @ C,"[{'Ni': '1.0'}, {'C': '1.0'}]","[Ni, C, NiC]"
2,Li2O2,"[{'Li': '2.0', 'O': '2.0'}]",[Li2O2]
3,Li2C8H4O4,"[{'Li': '2.0', 'C': '8.0', 'H': '4.0', 'O': '4...",[LiH2(C2O)2]
4,NVP / C,"[{'N': '1.0', 'V': '1.0', 'P': '1.0'}, {'C': '...","[VPN, C, VPCN]"
5,NiBr / Ni,"[{'Ni': '1.0', 'Br': '1.0'}, {'Ni': '1.0'}]","[NiBr, Ni, Ni2Br]"
6,Cu(OH)2,"[{'O': '2.0', 'H': '2.0', 'Cu': '1.0'}]","[Cu(HO)2, H2O2]"
7,CoSnO3,"[{'Co': '1.0', 'Sn': '1.0', 'O': '3.0'}]",[CoSnO3]
8,NiP2 @ C,"[{'Ni': '1.0', 'P': '2.0'}, {'C': '1.0'}]","[NiP2, C, NiP2C]"
9,SnSbCo,"[{'Sn': '1.0', 'Sb': '1.0', 'Co': '1.0'}]",[CoSnSb]


In [9]:
keep_cols = [
    "Property",
    "property_clean",
    "Name",
    "Value",
    "Unit",
    "Raw_value",
    "Raw_unit",
    "Warning",
    "Type",
    "DOI",
    "Title",
    "Journal",
    "Date",
    "Tag",
    "Info",
    "Specifier",
    "Num_records",
    "warning_has_L_or_R",
    "warning_has_S",
    "warning_clean_no_LR",
    "sodium_text_context",
    "cathode_text_context",
    "anode_or_electrolyte_context",
    "cde_formula_candidates",
]

df_cde_formula_records = (
    df_cde[keep_cols]
    .explode("cde_formula_candidates")
    .rename(columns={"cde_formula_candidates": "cde_reduced_formula"})
)

df_cde_formula_records = df_cde_formula_records[
    df_cde_formula_records["cde_reduced_formula"].notna()
].copy()

df_cde_formula_records["cde_formula_elements"] = df_cde_formula_records[
    "cde_reduced_formula"
].map(lambda f: ",".join(sorted(formula_elements(f))))

df_cde_formula_records["cde_formula_contains_Na"] = df_cde_formula_records[
    "cde_formula_elements"
].map(lambda s: "Na" in str(s).split(","))

df_cde_formula_records["cde_sodium_evidence"] = (
    df_cde_formula_records["cde_formula_contains_Na"]
    | df_cde_formula_records["sodium_text_context"]
)

df_cde_formula_records["cde_cathode_likely"] = (
    df_cde_formula_records["cathode_text_context"]
    | (
        df_cde_formula_records["cde_formula_contains_Na"]
        & ~df_cde_formula_records["anode_or_electrolyte_context"]
    )
)

df_cde_formula_records["doi_list"] = df_cde_formula_records["DOI"].map(split_doi_list)

print("Exploded CDE formula records:", df_cde_formula_records.shape)
print("Formula records containing Na:", int(df_cde_formula_records["cde_formula_contains_Na"].sum()))
print("Formula records with sodium evidence:", int(df_cde_formula_records["cde_sodium_evidence"].sum()))
print("Formula records likely cathode:", int(df_cde_formula_records["cde_cathode_likely"].sum()))

display(df_cde_formula_records.head())

Exploded CDE formula records: (354598, 29)
Formula records containing Na: 15391
Formula records with sodium evidence: 46953
Formula records likely cathode: 66602


,Property,property_clean,Name,Value,Unit,Raw_value,Raw_unit,Warning,Type,DOI,...,warning_clean_no_LR,sodium_text_context,cathode_text_context,anode_or_electrolyte_context,cde_reduced_formula,cde_formula_elements,cde_formula_contains_Na,cde_sodium_evidence,cde_cathode_likely,doi_list
0,Capacity,Capacity,Fe2O3 / C,659.0,Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),and 659,mAhg−1,"S,",NaN,10.1016/j.jallcom.2011.10.022,...,True,False,False,False,Fe2O3,"Fe,O",False,False,False,[10.1016/j.jallcom.2011.10.022]
0,Capacity,Capacity,Fe2O3 / C,659.0,Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),and 659,mAhg−1,"S,",NaN,10.1016/j.jallcom.2011.10.022,...,True,False,False,False,C,C,False,False,False,[10.1016/j.jallcom.2011.10.022]
0,Capacity,Capacity,Fe2O3 / C,659.0,Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),and 659,mAhg−1,"S,",NaN,10.1016/j.jallcom.2011.10.022,...,True,False,False,False,Fe2CO3,"C,Fe,O",False,False,False,[10.1016/j.jallcom.2011.10.022]
1,Coulombic Efficiency,Coulombic Efficiency,Ni @ C,100.0,Percent^(1.0),100,%,NaN,NaN,10.1039/C6TA02339H,...,True,False,False,False,Ni,Ni,False,False,False,[10.1039/C6TA02339H]
1,Coulombic Efficiency,Coulombic Efficiency,Ni @ C,100.0,Percent^(1.0),100,%,NaN,NaN,10.1039/C6TA02339H,...,True,False,False,False,C,C,False,False,False,[10.1039/C6TA02339H]


In [10]:
VALID_PROPERTIES = [
    "Capacity",
    "Voltage",
    "Energy",
    "Coulombic Efficiency",
    "Conductivity",
]

df_cde_formula_clean = df_cde_formula_records[
    df_cde_formula_records["property_clean"].isin(VALID_PROPERTIES)
    & df_cde_formula_records["Value"].notna()
    & df_cde_formula_records["warning_clean_no_LR"]
].copy()

df_cde_na_cathode_clean = df_cde_formula_clean[
    df_cde_formula_clean["cde_sodium_evidence"]
    & df_cde_formula_clean["cde_cathode_likely"]
].copy()

cde_clean_path = OUTPUT_DIR / "cde_formula_records_clean_no_LR.csv"
cde_na_cathode_path = OUTPUT_DIR / "cde_na_cathode_clean_filtered.csv"

df_cde_formula_clean.to_csv(cde_clean_path, index=False)
df_cde_na_cathode_clean.to_csv(cde_na_cathode_path, index=False)

print("Saved:", cde_clean_path)
print("Saved:", cde_na_cathode_path)

print("Clean formula records, no L/R:", df_cde_formula_clean.shape)
print("Na-cathode clean filtered records:", df_cde_na_cathode_clean.shape)

print("\nNa-cathode property counts:")
display(df_cde_na_cathode_clean["property_clean"].value_counts().to_frame("count"))

display(df_cde_na_cathode_clean.head())

Saved: sodium_cathode_cde_matching\outputs\cde_formula_records_clean_no_LR.csv
Saved: sodium_cathode_cde_matching\outputs\cde_na_cathode_clean_filtered.csv
Clean formula records, no L/R: (279505, 29)
Na-cathode clean filtered records: (13995, 29)

Na-cathode property counts:


,count
property_clean,
Capacity,7737
Voltage,4257
Energy,1170
Coulombic Efficiency,608
Conductivity,223


,Property,property_clean,Name,Value,Unit,Raw_value,Raw_unit,Warning,Type,DOI,...,warning_clean_no_LR,sodium_text_context,cathode_text_context,anode_or_electrolyte_context,cde_reduced_formula,cde_formula_elements,cde_formula_contains_Na,cde_sodium_evidence,cde_cathode_likely,doi_list
4,Capacity,Capacity,NVP / C,83.0,Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),"96.6 , 90.5 , 86.8 , 83.0 , 77.1 , 68.0 , and ...",mAhg−1,"S,",NaN,10.1016/j.ensm.2019.05.041,...,True,True,True,False,VPN,"N,P,V",False,True,True,[10.1016/j.ensm.2019.05.041]
4,Capacity,Capacity,NVP / C,83.0,Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),"96.6 , 90.5 , 86.8 , 83.0 , 77.1 , 68.0 , and ...",mAhg−1,"S,",NaN,10.1016/j.ensm.2019.05.041,...,True,True,True,False,C,C,False,True,True,[10.1016/j.ensm.2019.05.041]
4,Capacity,Capacity,NVP / C,83.0,Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),"96.6 , 90.5 , 86.8 , 83.0 , 77.1 , 68.0 , and ...",mAhg−1,"S,",NaN,10.1016/j.ensm.2019.05.041,...,True,True,True,False,VPCN,"C,N,P,V",False,True,True,[10.1016/j.ensm.2019.05.041]
12,Capacity,Capacity,NVP @ C,100.0,Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),"114 , 112 , 104 , 102 and 100",mAhg−1,"S,",NaN,10.1039/C7SE00537G,...,True,True,True,True,VPN,"N,P,V",False,True,True,[10.1039/C7SE00537G]
12,Capacity,Capacity,NVP @ C,100.0,Gram^(-1.0) Hour^(1.0) MilliAmpere^(1.0),"114 , 112 , 104 , 102 and 100",mAhg−1,"S,",NaN,10.1039/C7SE00537G,...,True,True,True,True,C,C,False,True,True,[10.1039/C7SE00537G]


In [11]:
def n_unique_dois(series):
    unique_dois = set()

    for item in series:
        if isinstance(item, list):
            unique_dois.update(item)

    return len(unique_dois)


def join_unique_values(series, max_items=5):
    values = []

    for x in series.dropna().astype(str):
        item = x.strip()

        if item and item not in values:
            values.append(item)

        if len(values) >= max_items:
            break

    return " | ".join(values)


df_cde_formula_property_agg = (
    df_cde_formula_clean
    .groupby(["cde_reduced_formula", "property_clean"], dropna=False)
    .agg(
        cde_value_median=("Value", "median"),
        cde_value_mean=("Value", "mean"),
        cde_value_min=("Value", "min"),
        cde_value_max=("Value", "max"),
        cde_record_count=("Value", "size"),
        cde_num_records_sum=("Num_records", "sum"),
        cde_doi_count=("doi_list", n_unique_dois),
        cde_cathode_context_count=("cathode_text_context", "sum"),
        cde_sodium_text_context_count=("sodium_text_context", "sum"),
        cde_warning_S_count=("warning_has_S", "sum"),
        cde_example_names=("Name", join_unique_values),
        cde_example_titles=("Title", join_unique_values),
        cde_example_journals=("Journal", join_unique_values),
    )
    .reset_index()
)

agg_path = OUTPUT_DIR / "cde_formula_property_aggregated_clean_no_LR.csv"
df_cde_formula_property_agg.to_csv(agg_path, index=False)

print("Saved:", agg_path)
print("Aggregated formula-property rows:", df_cde_formula_property_agg.shape)

display(df_cde_formula_property_agg.head())

Saved: sodium_cathode_cde_matching\outputs\cde_formula_property_aggregated_clean_no_LR.csv
Aggregated formula-property rows: (24552, 15)


,cde_reduced_formula,property_clean,cde_value_median,cde_value_mean,cde_value_min,cde_value_max,cde_record_count,cde_num_records_sum,cde_doi_count,cde_cathode_context_count,cde_sodium_text_context_count,cde_warning_S_count,cde_example_names,cde_example_titles,cde_example_journals
0,A,Capacity,478.850000,638.380000,132.500000,2430.000000,10,10,3,0,0,5,"Carbon A / CoPc | 5 C ( 3 A g−1 ), 10 C | LiFe...",LITHIUMAIRBATTERIESUSINGHYDROPHOBICROOMTEMPERA...,Journal of Power Sources | Electrochimica Acta
1,A,Energy,39.965333,39.965333,39.965333,39.965333,1,1,0,0,0,0,LiFePO4 / AS-0,,
2,A,Voltage,2.700000,2.441818,1.200000,3.700000,11,11,6,0,0,4,60-50 I 0 I00 50 0 .-Et4NPF4 / PC 0 -MeEt3NPF4...,EFFECTACTIVATEDCARBONELECTROLYTEPROPERTIESSUPE...,Transactions of Nonferrous Metals Society of C...
3,A1L1Li1Al0.05Co0.15Ni0.8C1N1O3,Voltage,1.975000,1.975000,1.550000,2.400000,2,2,1,0,0,0,Li4Ti5O12 / LiNi0.8Co0.15Al0.05O2LNCAO,Recent advances of Li 4 Ti 5 O 12 as a promisi...,Journal of Materials Chemistry A
4,A1Li1C1S1I5.4O2,Voltage,2.700000,2.700000,2.700000,2.700000,1,1,1,0,0,0,Li / LiAICI4.4SO2 / C,CYCLICVOLTAMMETRICRAMANSPECTROSCOPICSTUDIESLIL...,Journal of Power Sources


In [12]:
df_mp = df_mp.copy()
df_mp["mp_index"] = range(len(df_mp))

mp_formula_key_cols = [
    "formula_discharge",
    "formula_charge",
    "framework_formula",
    "formula_for_parsing",
    "battery_formula",
]

mp_formula_key_cols = [
    col for col in mp_formula_key_cols
    if col in df_mp.columns
]

mp_key_rows = []

for _, row in df_mp.iterrows():
    for col in mp_formula_key_cols:
        original_formula = row.get(col)

        reduced_formula = formula_to_reduced(original_formula)

        if reduced_formula:
            mp_key_rows.append({
                "mp_index": row["mp_index"],
                "mp_key_source": col,
                "mp_original_formula": original_formula,
                "mp_reduced_formula": reduced_formula,
            })

df_mp_formula_keys = pd.DataFrame(mp_key_rows).drop_duplicates()

mp_keys_path = OUTPUT_DIR / "mp_formula_keys_for_cde_matching.csv"
df_mp_formula_keys.to_csv(mp_keys_path, index=False)

print("Saved:", mp_keys_path)
print("MP formula keys:", df_mp_formula_keys.shape)
print("Unique MP reduced formulas:", df_mp_formula_keys["mp_reduced_formula"].nunique())

display(df_mp_formula_keys.head(20))

Saved: sodium_cathode_cde_matching\outputs\mp_formula_keys_for_cde_matching.csv
MP formula keys: (1664, 4)
Unique MP reduced formulas: 795


,mp_index,mp_key_source,mp_original_formula,mp_reduced_formula
0,0,formula_discharge,Na2MnPO4F,Na2MnPO4F
1,0,formula_charge,MnPO4F,MnPO4F
2,0,framework_formula,MnPO4F,MnPO4F
3,0,formula_for_parsing,Na2MnPO4F,Na2MnPO4F
4,1,formula_discharge,Na2MnP2O7,Na2MnP2O7
5,1,formula_charge,MnP2O7,MnP2O7
6,1,framework_formula,MnP2O7,MnP2O7
7,1,formula_for_parsing,Na2MnP2O7,Na2MnP2O7
8,2,formula_discharge,Na2MnCSO7,Na2MnCSO7
9,2,formula_charge,MnCSO7,MnCSO7


In [13]:
df_exact_matches_long = df_mp_formula_keys.merge(
    df_cde_formula_property_agg,
    left_on="mp_reduced_formula",
    right_on="cde_reduced_formula",
    how="left",
)

df_exact_matches_long = df_exact_matches_long[
    df_exact_matches_long["cde_record_count"].notna()
].copy()

exact_long_path = OUTPUT_DIR / "mp_cde_exact_formula_matches_long.csv"
df_exact_matches_long.to_csv(exact_long_path, index=False)

print("Saved:", exact_long_path)
print("Exact matched rows:", df_exact_matches_long.shape)
print("MP electrode records with exact CDE formula evidence:", df_exact_matches_long["mp_index"].nunique())

display(df_exact_matches_long.head())

Saved: sodium_cathode_cde_matching\outputs\mp_cde_exact_formula_matches_long.csv
Exact matched rows: (1395, 19)
MP electrode records with exact CDE formula evidence: 153


,mp_index,mp_key_source,mp_original_formula,mp_reduced_formula,cde_reduced_formula,property_clean,cde_value_median,cde_value_mean,cde_value_min,cde_value_max,cde_record_count,cde_num_records_sum,cde_doi_count,cde_cathode_context_count,cde_sodium_text_context_count,cde_warning_S_count,cde_example_names,cde_example_titles,cde_example_journals
0,0,formula_discharge,Na2MnPO4F,Na2MnPO4F,Na2MnPO4F,Capacity,113.00,111.917391,26.70000,249.4,46.0,50.0,9.0,44.0,34.0,26.0,Na2MnPO4F / C | Na2MnPO4F / C-rGO | Na2MnPO4F ...,Exploiting Na 2 MnPO 4 F as a high-capacity an...,RSC Advances | Ceramics International | Journa...
1,0,formula_discharge,Na2MnPO4F,Na2MnPO4F,Na2MnPO4F,Coulombic Efficiency,100.00,100.000000,100.00000,100.0,1.0,1.0,1.0,1.0,0.0,0.0,Na2MnPO4F / C,SYNTHESISCARBONCOATEDNA2MNPO4FHOLLOWSPHERESAPO...,Journal of Power Sources
2,0,formula_discharge,Na2MnPO4F,Na2MnPO4F,Na2MnPO4F,Energy,511.01,520.154950,371.36475,766.5,5.0,5.0,0.0,0.0,1.0,0.0,Na2MnPO4F / Na | Na2MnPO4F / C | Na2MnPO4F | N...,,
3,0,formula_discharge,Na2MnPO4F,Na2MnPO4F,Na2MnPO4F,Voltage,4.50,4.002143,1.00000,5.0,14.0,19.0,9.0,12.0,12.0,0.0,Na2MnPO4F | Na2MnPO4F / C-Na | Na2MnPO4F / Na ...,Exploiting Na 2 MnPO 4 F as a high-capacity an...,RSC Advances | Materials Letters | Energy Stor...
6,0,formula_for_parsing,Na2MnPO4F,Na2MnPO4F,Na2MnPO4F,Capacity,113.00,111.917391,26.70000,249.4,46.0,50.0,9.0,44.0,34.0,26.0,Na2MnPO4F / C | Na2MnPO4F / C-rGO | Na2MnPO4F ...,Exploiting Na 2 MnPO 4 F as a high-capacity an...,RSC Advances | Ceramics International | Journa...


In [14]:
if len(df_exact_matches_long) > 0:

    df_by_mp_property = (
        df_exact_matches_long
        .groupby(["mp_index", "property_clean"], dropna=False)
        .agg(
            cde_value_median=("cde_value_median", "median"),
            cde_record_count=("cde_record_count", "sum"),
            cde_doi_count=("cde_doi_count", "sum"),
            cde_cathode_context_count=("cde_cathode_context_count", "sum"),
            cde_sodium_text_context_count=("cde_sodium_text_context_count", "sum"),
            cde_warning_S_count=("cde_warning_S_count", "sum"),
        )
        .reset_index()
    )

    value_pivot = df_by_mp_property.pivot(
        index="mp_index",
        columns="property_clean",
        values="cde_value_median",
    )

    record_pivot = df_by_mp_property.pivot(
        index="mp_index",
        columns="property_clean",
        values="cde_record_count",
    )

    doi_pivot = df_by_mp_property.pivot(
        index="mp_index",
        columns="property_clean",
        values="cde_doi_count",
    )

    value_pivot.columns = [
        f"cde_{str(c).lower().replace(' ', '_')}_median"
        for c in value_pivot.columns
    ]

    record_pivot.columns = [
        f"cde_{str(c).lower().replace(' ', '_')}_record_count"
        for c in record_pivot.columns
    ]

    doi_pivot.columns = [
        f"cde_{str(c).lower().replace(' ', '_')}_doi_count"
        for c in doi_pivot.columns
    ]

    df_cde_wide_by_mp = pd.concat(
        [value_pivot, record_pivot, doi_pivot],
        axis=1,
    ).reset_index()

    df_match_summary = (
        df_exact_matches_long
        .groupby("mp_index")
        .agg(
            cde_exact_formula_match_count=(
                "cde_reduced_formula",
                lambda x: len(set(x.dropna())),
            ),
            cde_matched_reduced_formulas=(
                "cde_reduced_formula",
                lambda x: ",".join(sorted(set(x.dropna()))),
            ),
            cde_matched_mp_key_sources=(
                "mp_key_source",
                lambda x: ",".join(sorted(set(x.dropna()))),
            ),
            cde_total_record_count=("cde_record_count", "sum"),
            cde_total_doi_count=("cde_doi_count", "sum"),
            cde_total_cathode_context_count=("cde_cathode_context_count", "sum"),
            cde_total_sodium_context_count=("cde_sodium_text_context_count", "sum"),
            cde_total_series_warning_count=("cde_warning_S_count", "sum"),
        )
        .reset_index()
    )

    df_cde_evidence_by_mp = df_match_summary.merge(
        df_cde_wide_by_mp,
        on="mp_index",
        how="left",
    )

else:
    df_cde_evidence_by_mp = pd.DataFrame({"mp_index": []})

print("MP-level CDE evidence table:", df_cde_evidence_by_mp.shape)
display(df_cde_evidence_by_mp.head())

MP-level CDE evidence table: (153, 24)


,mp_index,cde_exact_formula_match_count,cde_matched_reduced_formulas,cde_matched_mp_key_sources,cde_total_record_count,cde_total_doi_count,cde_total_cathode_context_count,cde_total_sodium_context_count,cde_total_series_warning_count,cde_capacity_median,...,cde_capacity_record_count,cde_conductivity_record_count,cde_coulombic_efficiency_record_count,cde_energy_record_count,cde_voltage_record_count,cde_capacity_doi_count,cde_conductivity_doi_count,cde_coulombic_efficiency_doi_count,cde_energy_doi_count,cde_voltage_doi_count
0,0,1,Na2MnPO4F,"formula_discharge,formula_for_parsing",132.0,38.0,114.0,94.0,52.0,113.0,...,92.0,NaN,2.0,10.0,28.0,18.0,NaN,2.0,0.0,18.0
1,1,1,Na2MnP2O7,"formula_discharge,formula_for_parsing",48.0,28.0,48.0,14.0,26.0,83.7,...,30.0,NaN,4.0,2.0,12.0,12.0,NaN,2.0,2.0,12.0
2,3,1,Na3MnPCO7,"formula_discharge,formula_for_parsing",40.0,28.0,34.0,32.0,14.0,125.0,...,30.0,NaN,NaN,4.0,6.0,16.0,NaN,NaN,4.0,8.0
3,7,1,Na3FePCO7,"formula_discharge,formula_for_parsing",6.0,6.0,2.0,4.0,0.0,96.0,...,2.0,NaN,NaN,2.0,2.0,4.0,NaN,NaN,0.0,2.0
4,8,1,CuPO4,"formula_charge,framework_formula",6.0,2.0,0.0,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,6.0,NaN,NaN,NaN,NaN,2.0


In [15]:
df_mp_master = df_mp.merge(
    df_cde_evidence_by_mp,
    on="mp_index",
    how="left",
)

df_mp_master["cde_exact_formula_match"] = (
    df_mp_master["cde_exact_formula_match_count"]
    .fillna(0)
    .astype(int)
    > 0
)

df_mp_master["cde_has_capacity_evidence"] = (
    df_mp_master.get("cde_capacity_record_count", pd.Series(index=df_mp_master.index, dtype=float))
    .fillna(0)
    > 0
)

df_mp_master["cde_has_voltage_evidence"] = (
    df_mp_master.get("cde_voltage_record_count", pd.Series(index=df_mp_master.index, dtype=float))
    .fillna(0)
    > 0
)

df_mp_master["cde_has_capacity_and_voltage_evidence"] = (
    df_mp_master["cde_has_capacity_evidence"]
    & df_mp_master["cde_has_voltage_evidence"]
)

master_path = OUTPUT_DIR / "mp_na_electrodes_master_v1_with_cde_evidence.csv"
exact_evidence_path = OUTPUT_DIR / "mp_na_electrodes_with_cde_exact_evidence.csv"

df_mp_master.to_csv(master_path, index=False)
df_mp_master[df_mp_master["cde_exact_formula_match"]].to_csv(exact_evidence_path, index=False)

print("Saved:", master_path)
print("Saved:", exact_evidence_path)

print("Total MP records:", len(df_mp_master))
print("MP records with exact CDE evidence:", int(df_mp_master["cde_exact_formula_match"].sum()))
print("MP records with CDE capacity evidence:", int(df_mp_master["cde_has_capacity_evidence"].sum()))
print("MP records with CDE voltage evidence:", int(df_mp_master["cde_has_voltage_evidence"].sum()))
print("MP records with both capacity and voltage evidence:", int(df_mp_master["cde_has_capacity_and_voltage_evidence"].sum()))

if "conservative_earth_abundant_candidate" in df_mp_master.columns:
    conservative = df_mp_master["conservative_earth_abundant_candidate"] == True
    print(
        "Conservative candidates with exact CDE evidence:",
        int((conservative & df_mp_master["cde_exact_formula_match"]).sum()),
    )

display(
    df_mp_master[
        [
            "battery_formula",
            "formula_discharge",
            "framework_formula",
            "preliminary_family",
            "average_voltage",
            "capacity_grav",
            "energy_grav",
            "conservative_earth_abundant_candidate",
            "cde_exact_formula_match",
            "cde_matched_reduced_formulas",
            "cde_capacity_median",
            "cde_voltage_median",
            "cde_capacity_record_count",
            "cde_voltage_record_count",
            "cde_total_doi_count",
        ]
    ].head(30)
)

Saved: sodium_cathode_cde_matching\outputs\mp_na_electrodes_master_v1_with_cde_evidence.csv
Saved: sodium_cathode_cde_matching\outputs\mp_na_electrodes_with_cde_exact_evidence.csv
Total MP records: 416
MP records with exact CDE evidence: 153
MP records with CDE capacity evidence: 138
MP records with CDE voltage evidence: 146
MP records with both capacity and voltage evidence: 132
Conservative candidates with exact CDE evidence: 18


,battery_formula,formula_discharge,framework_formula,preliminary_family,average_voltage,capacity_grav,energy_grav,conservative_earth_abundant_candidate,cde_exact_formula_match,cde_matched_reduced_formulas,cde_capacity_median,cde_voltage_median,cde_capacity_record_count,cde_voltage_record_count,cde_total_doi_count
0,Na0-2MnPO4F,Na2MnPO4F,MnPO4F,phosphate_or_NASICON_like,3.864973,249.446804,964.105080,True,True,Na2MnPO4F,113.0000,4.5000,92.0,28.0,38.0
1,Na0-2MnP2O7,Na2MnP2O7,MnP2O7,phosphate_or_NASICON_like,4.058779,195.018501,791.536945,True,True,Na2MnP2O7,83.7000,3.5000,30.0,12.0,28.0
2,Na0-2MnCSO7,Na2MnCSO7,MnCSO7,sulfate_like,4.100875,208.580697,855.363301,True,False,NaN,NaN,NaN,NaN,NaN,NaN
3,Na1-3MnPCO7,Na3MnPCO7,MnPCO7,phosphate_or_NASICON_like,3.369558,192.202735,647.638332,True,True,Na3MnPCO7,125.0000,4.4000,30.0,6.0,28.0
4,Na0-4Fe2(PO4)3,Na4Fe2(PO4)3,Fe2(PO4)3,phosphate_or_NASICON_like,3.956552,219.431043,868.190282,True,False,NaN,NaN,NaN,NaN,NaN,NaN
5,Na0-2FeCSO7,Na2FeCSO7,FeCSO7,sulfate_like,3.892605,207.847172,809.067032,True,False,NaN,NaN,NaN,NaN,NaN,NaN
6,Na2-6Fe2C4SO16,Na6Fe2C4SO16,Fe2C4SO16,sulfate_like,3.904088,183.030590,714.567591,True,False,NaN,NaN,NaN,NaN,NaN,NaN
7,Na1-3FePCO7,Na3FePCO7,FePCO7,phosphate_or_NASICON_like,3.443900,191.579709,659.781296,True,True,Na3FePCO7,96.0000,4.4000,2.0,2.0,6.0
8,Na0-1CuPO4,NaCuPO4,CuPO4,phosphate_or_NASICON_like,3.827827,147.660761,565.219775,True,True,CuPO4,NaN,3.6000,NaN,6.0,2.0
9,Na0-6Mn3(PO4)4,Na6Mn3(PO4)4,Mn3(PO4)4,phosphate_or_NASICON_like,3.469876,235.569716,817.397652,True,False,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
audit_rows = []

def add_metric(name, value):
    audit_rows.append({"metric": name, "value": value})


add_metric("mp_database_version", mp_metadata.get("mp_database_version"))
add_metric("mp_total_records", len(df_mp_master))
add_metric("mp_basic_screening_pass", int(df_mp_master.get("basic_screening_pass", pd.Series(dtype=bool)).fillna(False).sum()))
add_metric("mp_hard_exclusion_free_candidates", int(df_mp_master.get("hard_exclusion_free_candidate", pd.Series(dtype=bool)).fillna(False).sum()))
add_metric("mp_conservative_earth_abundant_candidates", int(df_mp_master.get("conservative_earth_abundant_candidate", pd.Series(dtype=bool)).fillna(False).sum()))

add_metric("cde_merged_rows", len(df_cde))
add_metric("cde_formula_records", len(df_cde_formula_records))
add_metric("cde_formula_records_clean_no_LR", len(df_cde_formula_clean))
add_metric("cde_na_cathode_clean_filtered_records", len(df_cde_na_cathode_clean))
add_metric("cde_aggregated_formula_property_rows", len(df_cde_formula_property_agg))

add_metric("mp_records_with_exact_cde_formula_evidence", int(df_mp_master["cde_exact_formula_match"].sum()))
add_metric("mp_records_with_cde_capacity_evidence", int(df_mp_master["cde_has_capacity_evidence"].sum()))
add_metric("mp_records_with_cde_voltage_evidence", int(df_mp_master["cde_has_voltage_evidence"].sum()))
add_metric("mp_records_with_cde_capacity_and_voltage_evidence", int(df_mp_master["cde_has_capacity_and_voltage_evidence"].sum()))

if "conservative_earth_abundant_candidate" in df_mp_master.columns:
    conservative_mask = df_mp_master["conservative_earth_abundant_candidate"] == True
    add_metric("conservative_candidates_with_exact_cde_evidence", int((conservative_mask & df_mp_master["cde_exact_formula_match"]).sum()))
    add_metric("conservative_candidates_with_cde_capacity_evidence", int((conservative_mask & df_mp_master["cde_has_capacity_evidence"]).sum()))
    add_metric("conservative_candidates_with_cde_voltage_evidence", int((conservative_mask & df_mp_master["cde_has_voltage_evidence"]).sum()))

df_audit = pd.DataFrame(audit_rows)

audit_path = AUDIT_DIR / "cde_matching_audit_summary.csv"
df_audit.to_csv(audit_path, index=False)

print("Saved:", audit_path)
display(df_audit)

Saved: sodium_cathode_cde_matching\audit\cde_matching_audit_summary.csv


,metric,value
0,mp_database_version,2026.04.13
1,mp_total_records,416
2,mp_basic_screening_pass,164
3,mp_hard_exclusion_free_candidates,112
4,mp_conservative_earth_abundant_candidates,56
5,cde_merged_rows,214617
6,cde_formula_records,354598
7,cde_formula_records_clean_no_LR,279505
8,cde_na_cathode_clean_filtered_records,13995
9,cde_aggregated_formula_property_rows,24552


In [17]:
candidate_cols = [
    "battery_formula",
    "formula_discharge",
    "framework_formula",
    "preliminary_family",
    "average_voltage",
    "capacity_grav",
    "energy_grav",
    "max_delta_volume",
    "stability_charge",
    "stability_discharge",
    "mp_preliminary_score",
    "conservative_earth_abundant_candidate",
    "hard_exclusion_free_candidate",
    "cde_exact_formula_match",
    "cde_matched_reduced_formulas",
    "cde_capacity_median",
    "cde_voltage_median",
    "cde_energy_median",
    "cde_capacity_record_count",
    "cde_voltage_record_count",
    "cde_total_doi_count",
]

candidate_cols = [c for c in candidate_cols if c in df_mp_master.columns]

df_top_evidence = df_mp_master[
    (df_mp_master.get("conservative_earth_abundant_candidate", False) == True)
    & (df_mp_master["cde_exact_formula_match"] == True)
].copy()

sort_cols = [c for c in ["mp_preliminary_score", "cde_total_doi_count"] if c in df_top_evidence.columns]

if sort_cols:
    df_top_evidence = df_top_evidence.sort_values(
        by=sort_cols,
        ascending=[False] * len(sort_cols),
    )

top_evidence_path = OUTPUT_DIR / "top_conservative_candidates_with_cde_evidence.csv"
df_top_evidence[candidate_cols].to_csv(top_evidence_path, index=False)

print("Saved:", top_evidence_path)
print("Top conservative candidates with exact CDE evidence:", len(df_top_evidence))

display(df_top_evidence[candidate_cols].head(50))

Saved: sodium_cathode_cde_matching\outputs\top_conservative_candidates_with_cde_evidence.csv
Top conservative candidates with exact CDE evidence: 18


,battery_formula,formula_discharge,framework_formula,preliminary_family,average_voltage,capacity_grav,energy_grav,max_delta_volume,stability_charge,stability_discharge,...,conservative_earth_abundant_candidate,hard_exclusion_free_candidate,cde_exact_formula_match,cde_matched_reduced_formulas,cde_capacity_median,cde_voltage_median,cde_energy_median,cde_capacity_record_count,cde_voltage_record_count,cde_total_doi_count
0,Na0-2MnPO4F,Na2MnPO4F,MnPO4F,phosphate_or_NASICON_like,3.864973,249.446804,964.105080,0.286562,0.035308,0.000000,...,True,True,True,Na2MnPO4F,113.0000,4.5000,511.010000,92.0,28.0,38.0
1,Na0-2MnP2O7,Na2MnP2O7,MnP2O7,phosphate_or_NASICON_like,4.058779,195.018501,791.536945,0.123101,0.072623,0.000365,...,True,True,True,Na2MnP2O7,83.7000,3.5000,300.000000,30.0,12.0,28.0
3,Na1-3MnPCO7,Na3MnPCO7,MnPCO7,phosphate_or_NASICON_like,3.369558,192.202735,647.638332,0.073266,0.000000,0.000000,...,True,True,True,Na3MnPCO7,125.0000,4.4000,467.480882,30.0,6.0,28.0
7,Na1-3FePCO7,Na3FePCO7,FePCO7,phosphate_or_NASICON_like,3.443900,191.579709,659.781296,0.010975,0.040622,0.000000,...,True,True,True,Na3FePCO7,96.0000,4.4000,405.120000,2.0,2.0,6.0
8,Na0-1CuPO4,NaCuPO4,CuPO4,phosphate_or_NASICON_like,3.827827,147.660761,565.219775,0.119355,0.005488,0.027283,...,True,True,True,CuPO4,NaN,3.6000,NaN,NaN,6.0,2.0
11,Na0-1MnO2,NaMnO2,MnO2,transition_metal_oxide_like,2.745589,243.812486,669.408995,0.075650,0.013759,0.000000,...,True,True,True,"MnO2,NaMnO2",312.0000,2.2600,432.616982,2732.0,1232.0,2166.0
12,Na0-1MnO2,NaMnO2,MnO2,transition_metal_oxide_like,2.692695,243.812486,656.512717,0.163210,0.015144,0.014262,...,True,True,True,"MnO2,NaMnO2",312.0000,2.2600,432.616982,2732.0,1232.0,2166.0
13,Na0-1MnPO4,NaMnPO4,MnPO4,phosphate_or_NASICON_like,3.387074,155.012197,525.037724,0.173845,0.002637,0.000000,...,True,True,True,"MnPO4,NaMnPO4",107.4625,3.6000,185.550000,48.0,18.0,28.0
14,Na0-1MnO2,NaMnO2,MnO2,transition_metal_oxide_like,2.681036,243.812486,653.669979,0.281759,0.032930,0.030517,...,True,True,True,"MnO2,NaMnO2",312.0000,2.2600,432.616982,2732.0,1232.0,2166.0
15,Na0-1MnPO4,NaMnPO4,MnPO4,phosphate_or_NASICON_like,3.297378,155.012197,511.133787,0.205882,0.000000,0.010554,...,True,True,True,"MnPO4,NaMnPO4",107.4625,3.6000,185.550000,48.0,18.0,28.0


In [18]:
expected_outputs = [
    OUTPUT_DIR / "cde_formula_records_clean_no_LR.csv",
    OUTPUT_DIR / "cde_na_cathode_clean_filtered.csv",
    OUTPUT_DIR / "cde_formula_property_aggregated_clean_no_LR.csv",
    OUTPUT_DIR / "mp_formula_keys_for_cde_matching.csv",
    OUTPUT_DIR / "mp_cde_exact_formula_matches_long.csv",
    OUTPUT_DIR / "mp_na_electrodes_master_v1_with_cde_evidence.csv",
    OUTPUT_DIR / "mp_na_electrodes_with_cde_exact_evidence.csv",
    OUTPUT_DIR / "top_conservative_candidates_with_cde_evidence.csv",
    AUDIT_DIR / "cde_matching_audit_summary.csv",
]

print("Final output file audit:")

for path in expected_outputs:
    status = "FOUND" if path.exists() else "MISSING"
    size_mb = path.stat().st_size / (1024 * 1024) if path.exists() else 0
    print(f"{status:7s} | {size_mb:8.2f} MB | {path}")

Final output file audit:
FOUND   |   113.38 MB | sodium_cathode_cde_matching\outputs\cde_formula_records_clean_no_LR.csv
FOUND   |     6.13 MB | sodium_cathode_cde_matching\outputs\cde_na_cathode_clean_filtered.csv
FOUND   |     6.98 MB | sodium_cathode_cde_matching\outputs\cde_formula_property_aggregated_clean_no_LR.csv
FOUND   |     0.06 MB | sodium_cathode_cde_matching\outputs\mp_formula_keys_for_cde_matching.csv
FOUND   |     0.81 MB | sodium_cathode_cde_matching\outputs\mp_cde_exact_formula_matches_long.csv
FOUND   |    23.06 MB | sodium_cathode_cde_matching\outputs\mp_na_electrodes_master_v1_with_cde_evidence.csv
FOUND   |     8.23 MB | sodium_cathode_cde_matching\outputs\mp_na_electrodes_with_cde_exact_evidence.csv
FOUND   |     0.00 MB | sodium_cathode_cde_matching\outputs\top_conservative_candidates_with_cde_evidence.csv
FOUND   |     0.00 MB | sodium_cathode_cde_matching\audit\cde_matching_audit_summary.csv
